In [ ]:
อันนี้เป็นการเปรียบเทียบชุดข้อมูล

In [ ]:
OLD (no group split — data leak)
X, y = [], []
for folder in ["processed", "augmented"]:        # ← loads BOTH together
    for label_idx, sign in enumerate(TARGET_SIGNS):
        for npy_file in os.listdir(.../folder/sign):
            X.append(np.load(...))
            y.append(label_idx)

X, y = np.array(X), np.array(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y   # ← split AFTER mixing
)
ซึ่งก็คือเกิด Overfitting ครับ ตอนใช้งานจริงก็บ่งครับ

In [ ]:
ด้วยความที่ว่า คลิปอย่างเช่น คลิป คุณ กับอีก 3augment variation ที่มีการเปลี่ยนแปลงที่ใกล้เคียงมากๆ แม้จะดัดแปลงบางส่วน
ข้อมูลที่แยกมาก็จะทรงประมาณนี้
TRAIN gets:  คุณ_1_aug_0, คุณ_1_aug_2, คุณ_1_aug_3
TEST  gets:  คุณ_1_aug_4   ← a near-twin of aug_2 ซึ่งมันควรจะเป็นตัวเทรน ไม่ใช่test

In [ ]:
NEW (group-aware — honest)
# 1) gather ORIGINALS only (processed, no augmented)
originals = [(path, label, sign, stem) for each .npy in processed/]

# 2) split the ORIGINALS first
train_orig, test_orig = train_test_split(originals, test_size=0.2,
                                         random_state=42, stratify=labels)

# 3) TEST = originals only
X_test = [load(o) for o in test_orig]

# 4) TRAIN = train originals + ONLY their augmented copies
train_keys = {(sign, stem) for o in train_orig}
for aug_file in augmented/:
    orig_stem = aug_file.rsplit("_aug_", 1)[0]
    if (sign, orig_stem) in train_keys:      # ← only if its parent is in train
        X_train.append(load(aug_file))

In [ ]:
If คุณ_1 → TRAIN:  then คุณ_1 + all its aug_0..aug_4 go to TRAIN
If คุณ_1 → TEST:   then ONLY คุณ_1 (the clean original) goes to TEST — no augments